# FM Comparison — segmentation qualitative gallery

For each segmentation dataset, compare every backbone's UPerNet prediction on the same example images (red = prediction, cyan outline = GT, per-cell Dice). Trains one deployment head per backbone and persists it under `OUTPUT_DIR/upernet_heads/` so re-runs skip the training step.

This notebook is sibling to `fm_comparison_seg.ipynb` — it shares the same backbone factory + dataset builder + evaluator config so the heads it trains match what the main benchmark would have trained.

## Imports

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from radharmony.evaluator import (
    LinearProbeSegEvaluator,
    ConvProbeSegEvaluator,
    UPerNetSegEvaluator,
)

## Configuration

Edit the `*_TO_RUN` lists to control what executes. Verification phase: keep them minimal until the schemas check out.

In [ ]:
OUTPUT_DIR  = Path("./eval_out/fm_comparison_seg/")
RESULTS_DIR = OUTPUT_DIR / "results"
# Same convention as notebooks/backbones/done/*: per-(backbone × dataset) subdir under one cache root.
# The MONAI PersistentDataset dir lives here (segmentation evaluator does not cache embeddings).
CACHE_ROOT  = Path("/path/to/cache/")
for d in (OUTPUT_DIR, RESULTS_DIR, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0"

# ─── what to run ──────────────────────────────────────
BACKBONES_TO_RUN = [
    # vision-only
    "rad_dino", "dinov3", "chexfound", "ark_plus", "medical_mae", "eva_x",
    # vision-language (image tower only — segmentation uses dense features, not text)
    "biomed_clip", "chexagent", "medsiglip", "medimageinsights", "siglip2",
]
PROBES_TO_RUN    = ["linear_seg", "conv_probe_seg", "upernet_seg"]  # segmentation heads to run
DATASETS_TO_RUN  = ["siim_acr_ptx", "montgomery_cxr"]  # extend with more seg datasets as builders are wired below
# ────────────────────────────────────────────────────────────

# K-fold protocol (variance comes from the fold axis)
N_FOLDS         = 5
N_TRAIN_SAMPLES = None             # None = full fold train split; [200, 500, 1500] = sweep

# Training
EPOCHS          = 20
LR              = 1e-3
WEIGHT_DECAY    = 0.05

# Early stopping (forwarded into all three seg evaluators)
# - Val signal: both k-fold and fixed-split modes carve a VAL_FRACTION
#   slice from the train subsample. The held-out fold (k-fold) and the
#   test set (fixed-split) are used only for scoring.
# - Set VAL_FRACTION=0.0 to disable early stopping.
EARLY_STOP_METRIC   = "dice"       # any key in the seg metric panel: dice|iou|pixel_acc|tpr|tnr|ppv|npv
EARLY_STOP_PATIENCE = 5
VAL_FRACTION        = 0.1

# Segmentation head
NUM_CLASSES     = 2                # binary (background + foreground)

# Per-dataset class weights: foreground sparsity differs by task.
# SIIM-ACR PTX foreground ≈ 1–3% of pixels → upweight foreground.
# Montgomery lung masks ≈ 30–40% of pixels → near-balanced.
CLASS_WEIGHTS = {
    "siim_acr_ptx":   [1.0, 5.0],
    "montgomery_cxr": [1.0, 1.0],
}

# Per-dataset batch size: smaller corpora can use a smaller batch to fit
# bigger crops in memory; large corpora benefit from larger batches.
# Note: UPerNet's PSP module fails at batch_size=1 (BatchNorm on pool-to-1×1
# branch); keep all entries >=2.
BATCH_SIZE = {
    "siim_acr_ptx":   16,
    "montgomery_cxr": 8,
}

# Common
BASE_SEED       = 0
NUM_WORKERS     = 4
AUTOCAST_DTYPE  = torch.bfloat16
SAVE_PREDICTIONS = False           # set True to dump GT/Pred/Prob PNGs under RESULTS_DIR


## Backbone registry

Each entry is `(factory_callable, factory_kwargs, has_text_encoder)`. For segmentation we always pass `output_keys={"img", "mask"}`; vision-language factories still return their 4-tuple but we only keep the image encoder.

In [ ]:
from radharmony.evaluator.backbones import (
    make_raddino, make_dinov3, make_chexfound, make_ark_plus,
    make_medical_mae, make_eva_x,
    make_biomed_clip, make_chexagent, make_medsiglip,
    make_medimageinsights, make_siglip2,
)

SEG_KWARGS = {"device": DEVICE, "output_keys": {"img", "mask"}}

BACKBONE_FACTORIES: dict = {
    # vision-only
    "rad_dino":         (make_raddino,         SEG_KWARGS, False),
    "dinov3":           (make_dinov3,          SEG_KWARGS, False),
    "chexfound":        (make_chexfound,       SEG_KWARGS, False),
    "ark_plus":         (make_ark_plus,        SEG_KWARGS, False),
    "medical_mae":      (make_medical_mae,     SEG_KWARGS, False),
    "eva_x":            (make_eva_x,           SEG_KWARGS, False),
    # vision-language (we keep only the image encoder for segmentation)
    "biomed_clip":      (make_biomed_clip,     SEG_KWARGS, True),
    "chexagent":        (make_chexagent,       SEG_KWARGS, True),
    "medsiglip":        (make_medsiglip,       SEG_KWARGS, True),
    "medimageinsights": (make_medimageinsights,SEG_KWARGS, True),
    "siglip2":          (make_siglip2,         SEG_KWARGS, True),
}

## Dataset builders

Each builder takes a backbone-specific `transform` (which includes the mask preprocess step) and returns the RadHarmony dataset with `output_mask=True`. NAS paths are inlined; edit for your environment.

To add another segmentation dataset:
1. Add a builder here that returns a RadHarmony dataset with `output_mask=True`.
2. Add its key (e.g. `"ranzcr_clip"`) to `DATASETS_TO_RUN`.
3. Wire it into the `build_dataset` dispatcher below.

In [ ]:
from radharmony.dataset import SIIMACRPTXDataset, MontgomeryCXRDataset
from radharmony.harmonizer import MontgomeryCXRHarmonizer

SIIM_IMAGE_DIR = "/path/to/SIIM_ACR_Pneumothorax/dicom-images-train/"
SIIM_MASK_DIR  = "/path/to/SIIM_ACR_Pneumothorax/masks/"

# Montgomery — fused (left ∪ right) lung mask per image.
# We restrict to TB-positive cases (58 of 138) per study design.
MONT_IMAGE_DIR = "/path/to/Montgomery-CXR/MontgomerySet/CXR_png/"
MONT_MASK_DIR  = "/path/to/mask_verify_out/montgomery_fused/"


def build_siim_acr_ptx(transform, cache_dir=None):
    return SIIMACRPTXDataset(
        base_image_dir=SIIM_IMAGE_DIR,
        mask_output_dir=SIIM_MASK_DIR,
        transform=transform,
        cache_dir=cache_dir,
        output_mask=True,
    )


def build_montgomery_cxr(transform, cache_dir=None):
    """Montgomery CXR — TB-positive cases only (lung-mask segmentation)."""
    h = MontgomeryCXRHarmonizer(base_dir=MONT_IMAGE_DIR)
    df = h.harmonize(mask_output_dir=MONT_MASK_DIR)
    df_pos = df[df["tuberculosis"] == 1].reset_index(drop=True)
    print(f"[montgomery_cxr] TB-positive only: {len(df_pos):,} of {len(df):,} cases")
    return MontgomeryCXRDataset(
        base_image_dir=MONT_IMAGE_DIR,
        transform=transform,
        cache_dir=cache_dir,
        output_mask=True,
        harmonized_df=df_pos,
    )

# ── extend here ───────────────────────────────────────────────────────────
# from radharmony.dataset import RANZCRClipDataset
# RANZCR_IMAGE_DIR = "/path/to/ranzcr/train/"
# RANZCR_MASK_DIR  = "/path/to/ranzcr/masks/"
# RANZCR_ANN_CSV   = "/path/to/ranzcr/train_annotations.csv"
# def build_ranzcr_clip(transform, cache_dir=None):
#     return RANZCRClipDataset(
#         base_image_dir=RANZCR_IMAGE_DIR,
#         mask_output_dir=RANZCR_MASK_DIR,
#         annotations_csv_path=RANZCR_ANN_CSV,
#         transform=transform,
#         cache_dir=cache_dir,
#         output_mask=True,
#     )

DATASET_BUILDERS = {
    "siim_acr_ptx":   build_siim_acr_ptx,
    "montgomery_cxr": build_montgomery_cxr,
    # "ranzcr_clip": build_ranzcr_clip,
}

## Probe runners

Each `run_*_seg` issues a **k-fold** call using `GroupKFold(n_folds)` on `patient_id`. The frozen encoder is re-applied per batch (no embedding cache — segmentation needs dense per-pixel features, which are too large to cache to disk like a single CLS token).

The three runners share `_COMMON_SEG_KWARGS` so the early-stopping knobs (`EARLY_STOP_METRIC`, `EARLY_STOP_PATIENCE`, `VAL_FRACTION`) and training hyper-params are forwarded to all heads identically.

In [ ]:
def _has_result(dataset_key, probe_key, backbone_key):
    return (RESULTS_DIR / dataset_key / probe_key / backbone_key / "results.csv").exists()


def _output_dir(dataset_key, probe_key, backbone_key):
    p = RESULTS_DIR / dataset_key / probe_key / backbone_key
    p.mkdir(parents=True, exist_ok=True)
    return str(p)


def _monai_cache(backbone_key, dataset_key):
    return str(CACHE_ROOT / f"{backbone_key}_{dataset_key}")

In [ ]:
_COMMON_SEG_KWARGS = dict(
    num_classes=NUM_CLASSES,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_predictions=SAVE_PREDICTIONS,
    num_workers=NUM_WORKERS,
    device=DEVICE,
    autocast_dtype=AUTOCAST_DTYPE,
    base_seed=BASE_SEED,
    early_stop_metric=EARLY_STOP_METRIC,
    early_stop_patience=EARLY_STOP_PATIENCE,
    val_fraction=VAL_FRACTION,
)


def _run_seg(cls, probe_key, image_encoder, dataset, *, backbone_key, dataset_key, **extra):
    ev = cls(
        image_encoder=image_encoder,
        dataset=dataset,
        class_weights=CLASS_WEIGHTS[dataset_key],
        batch_size=BATCH_SIZE[dataset_key],
        output_dir=_output_dir(dataset_key, probe_key, backbone_key),
        **_COMMON_SEG_KWARGS,
        **extra,
    )
    df = ev.evaluate()
    ev.save_results(df)
    return df


def run_linear_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(LinearProbeSegEvaluator, "linear_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


def run_conv_probe_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(ConvProbeSegEvaluator, "conv_probe_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


def run_upernet_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(UPerNetSegEvaluator, "upernet_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


PROBE_RUNNERS = {
    "linear_seg":     run_linear_seg,
    "conv_probe_seg": run_conv_probe_seg,
    "upernet_seg":    run_upernet_seg,
}


## Display name helper

Tiny `dn(...)` lookup used in the gallery titles.

In [ ]:
DISPLAY_NAMES = {
    'rad_dino': 'RAD-DINO', 'dinov3': 'DINOv3', 'chexfound': 'CheXFound',
    'ark_plus': 'Ark+', 'medical_mae': 'Medical MAE', 'eva_x': 'EVA-X',
    'biomed_clip': 'BiomedCLIP', 'chexagent': 'CheXagent',
    'medsiglip': 'MedSigLIP', 'medimageinsights': 'MedImageInsights',
    'siglip2': 'SigLIP 2',
    'siim_acr_ptx': 'SIIM-ACR-PTX', 'montgomery_cxr': 'Montgomery',
}

def dn(x):
    return DISPLAY_NAMES.get(x, x)

## Qualitative results — UPerNet predictions across backbones

For each dataset, compare **every backbone's** UPerNet prediction on the same example images: columns are example images, the top row is ground truth, each following row is one backbone (red fill = prediction, cyan outline = ground truth, per-cell **Dice**).

This uses the segmentation evaluator's first-class API: `fit()` trains a **deployment head on all data**, `save_head` / `load_head` persist just the trainable head (reused on re-run, so you train once), and `predict(..., return_images=True)` returns masks + the input image for overlays. Heads are saved under `OUTPUT_DIR/upernet_heads/`; set `VIZ_FORCE_RETRAIN = True` to retrain.

Note: the saved head is trained on **all** images (the deployment model you keep for the future), so the examples shown are *in-sample* — this is a qualitative gallery of what each model produces, not a held-out evaluation (for held-out Dice see the benchmark tables above). First run trains one head per backbone (heavy); trim `VIZ_BACKBONES` or lower `VIZ_EPOCHS` for a quick look. Columns are the images with the most ground-truth foreground (sparse SIIM).

In [ ]:
# Compare every backbone's UPerNet prediction on the same example images, using
# the evaluator's first-class API (no bespoke training/inference glue):
#   fit()  -> trains a deployment head on all data MINUS a patient-grouped
#             inner-val slice that is held out for early stopping
#   inner_val_indices() -> the exact images in that slice (deterministic from
#             base_seed; works even after load_head). We visualize on THESE,
#             not on training images, so the overlays reflect held-out
#             generalization rather than train-set fit. (Mild optimism remains:
#             the slice drove early-stopping epoch selection. For Dice that
#             exactly matches the benchmark table, use evaluate()'s OOF preds.)
#   save_head / load_head -> persist + reuse the head (skips retraining on re-run)
#   predict(..., return_images=True) -> masks + input image for overlays
# Grid: columns = examples, top row = GT, one row per backbone
# (red = pred, cyan outline = GT, per-cell Dice).
from torch.utils.data import Subset

VIZ_BACKBONES     = BACKBONES_TO_RUN   # all of them; trim e.g. ["rad_dino","chexfound","ark_plus"] for speed
VIZ_DATASETS      = DATASETS_TO_RUN
VIZ_N_EXAMPLES    = 3                  # columns (example images) per dataset
VIZ_EPOCHS        = EPOCHS              # lower (e.g. 8) for faster, rougher heads
VIZ_FORCE_RETRAIN = False               # True = retrain and overwrite saved heads

CKPT_DIR = OUTPUT_DIR / "upernet_heads"  # deployment heads persist here (save_head / load_head)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def _ckpt_path(dataset_key, backbone_key):
    return CKPT_DIR / f"{dataset_key}__{backbone_key}__upernet.pt"


def _make_evaluator(encoder, ds, dataset_key):
    return UPerNetSegEvaluator(
        image_encoder=encoder, dataset=ds,
        class_weights=CLASS_WEIGHTS[dataset_key], batch_size=BATCH_SIZE[dataset_key],
        num_classes=NUM_CLASSES, n_folds=N_FOLDS, epochs=VIZ_EPOCHS,
        lr=LR, weight_decay=WEIGHT_DECAY, device=DEVICE, autocast_dtype=AUTOCAST_DTYPE,
        base_seed=BASE_SEED, num_workers=NUM_WORKERS,
        early_stop_metric=EARLY_STOP_METRIC, early_stop_patience=EARLY_STOP_PATIENCE,
        val_fraction=VAL_FRACTION,
    )


def _choose_examples(n, candidates):
    """n randomly chosen candidate indices (seeded by BASE_SEED so the panel is
    reproducible across re-runs; drop the seed for a fresh draw each time)."""
    cand = [int(i) for i in candidates]
    #rng = np.random.default_rng(BASE_SEED)
    rng = np.random.default_rng(5566)
    return [int(i) for i in rng.choice(cand, size=min(n, len(cand)), replace=False)]


def _dice(a, b):
    a, b = a > 0, b > 0
    denom = int(a.sum() + b.sum())
    return 1.0 if denom == 0 else 2.0 * int((a & b).sum()) / denom


def _panel(ax, img, overlay=None, contour=None):
    ax.imshow(img, cmap="gray")
    if overlay is not None:
        ax.imshow(np.ma.masked_where(overlay == 0, overlay), cmap="autumn", alpha=0.5, vmin=0, vmax=1)
    if contour is not None and (contour > 0).any():
        ax.contour((contour > 0).astype(float), levels=[0.5], colors="cyan", linewidths=1.0)
    ax.set_xticks([])
    ax.set_yticks([])


def visualize_all_upernet(dataset_key, backbones=VIZ_BACKBONES, n_examples=VIZ_N_EXAMPLES):
    print(f"\n[viz] {dataset_key}: UPerNet x {len(backbones)} backbones (early-stopping val images)")
    example_idx = None
    rows = []          # (backbone_key, [per-example predict() dicts])
    ref = None         # first backbone's preds -> ground-truth reference row

    for bb in backbones:
        enc = ev = ds = None
        try:
            print(f"  - {bb}")
            factory, kwargs, _ = BACKBONE_FACTORIES[bb]
            transform, enc = factory(**kwargs)[:2]   # drop text tower if returned
            ds = DATASET_BUILDERS[dataset_key](transform, cache_dir=_monai_cache(bb, dataset_key))
            ev = _make_evaluator(enc, ds, dataset_key)

            ckpt = _ckpt_path(dataset_key, bb)
            if ckpt.exists() and not VIZ_FORCE_RETRAIN:
                ev.load_head(str(ckpt))
                print(f"    loaded {ckpt.name}")
            else:
                ev.fit()                      # head trained on all data minus the inner-val slice
                ev.save_head(str(ckpt))
                print(f"    saved  {ckpt.name}")

            if example_idx is None:           # choose columns once (shared across backbones)
                # The early-stopping val slice is held out from gradient training and
                # deterministic from base_seed, so it's identical for every backbone.
                val_idx = ev.inner_val_indices(ds)
                if len(val_idx) == 0:
                    print("    no inner-val slice (VAL_FRACTION=0 or too few patients) — skipping dataset")
                    return
                example_idx = _choose_examples(min(n_examples, len(val_idx)), val_idx)

            preds = ev.predict(ds, indices=example_idx, return_images=True)
            rows.append((bb, preds))
            if ref is None:
                ref = preds
        except Exception as e:
            print(f"  ! skip {bb}: {type(e).__name__}: {e}")
        del enc, ev, ds
        gc.collect()
        torch.cuda.empty_cache()

    if not rows or not example_idx:
        print(f"  nothing to show for {dataset_key}")
        return

    n_rows = len(example_idx)
    n_cols = len(rows) + 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = np.asarray(axes).reshape(n_rows, n_cols)

    # ground-truth column (reference backbone's image + GT)
    for r in range(n_rows):
        _panel(axes[r, 0], ref[r]["image"], overlay=ref[r]["y_true"])
        axes[r, 0].set_ylabel(f"val example {r + 1}", rotation=0, ha="right", va="center", fontsize=10)
    axes[0, 0].set_title("ground truth", fontsize=10)

    # one column per backbone (pred fill + GT outline + Dice)
    for c, (bb, preds) in enumerate(rows, start=1):
        for r in range(n_rows):
            p = preds[r]
            _panel(axes[r, c], p["image"], overlay=p["y_pred"], contour=p["y_true"])
            axes[r, c].text(
                0.03, 0.97, f"Dice {_dice(p['y_true'], p['y_pred']):.2f}",
                transform=axes[r, c].transAxes, va="top", ha="left", fontsize=8, color="white",
                bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"),
            )
        axes[0, c].set_title(dn(bb), fontsize=10)

    fig.suptitle(f"{dn(dataset_key)} — UPerNet across backbones  •  held-out early-stopping val images "
                 f"(red = pred, cyan = GT)", y=1.0, fontsize=14)
    plt.tight_layout()
    plt.show()


for _ds in VIZ_DATASETS:
    visualize_all_upernet(_ds)